# PixiJoy Depth (M2) — quality train (CPU OK)

**No GPU required.** Works on Colab CPU / High-RAM and on a laptop.

1. Runtime → accelerator **None** (or GPU if available) → **Run all**
2. On Colab, Drive mounts automatically to resume after disconnect
3. On a laptop: `cd` into the lab folder and run the same cells / CLI below

Drive path (Colab only): `MyDrive/pixijoy_ml/depth/`

Pin TFLite only if verify prints **OK** / **SHIP_OK**.

If you still see `assert ... GPU`, you have a **cached old notebook** — File → Open notebook from GitHub again (or Runtime → Disconnect and delete runtime), then hard-refresh the tab.


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TF", tf.__version__, "GPUs", gpus or "none (CPU OK)")
print('accelerator=', 'GPU' if gpus else 'CPU')
# Never assert on GPU — quality train is valid on CPU/laptop.


In [ ]:
import os, sys, urllib.request, zipfile, shutil
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = Path('/content').is_dir()

USE_DRIVE = IN_COLAB  # Drive only in Colab; laptop uses local ./dist + ./data

ROOT = Path('/content/pixijoy_depth') if IN_COLAB else Path('.').resolve()
DRIVE = Path('/content/drive/MyDrive/pixijoy_ml/depth')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print('cwd', ROOT, 'IN_COLAB', IN_COLAB, 'USE_DRIVE', USE_DRIVE)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    (DRIVE / 'dist').mkdir(parents=True, exist_ok=True)
    (DRIVE / 'data').mkdir(parents=True, exist_ok=True)

def _link_to_drive(name: str) -> None:
    if not USE_DRIVE:
        Path(name).mkdir(parents=True, exist_ok=True)
        return
    local = Path(name)
    target = DRIVE / name
    target.mkdir(parents=True, exist_ok=True)
    if local.exists() or local.is_symlink():
        if local.is_symlink():
            if local.resolve() == target.resolve():
                print(f'{name} -> Drive OK')
                return
            local.unlink()
        elif local.is_dir():
            for p in local.rglob('*'):
                if not p.is_file():
                    continue
                dest = target / p.relative_to(local)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.is_file():
                    shutil.copy2(p, dest)
            shutil.rmtree(local)
        else:
            local.unlink()
    local.symlink_to(target)
    print(f'{name} -> {target}')

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/depth_lab.zip'
ok = Path('train.py').is_file() and Path('prepare_quality_data.py').is_file()
if not ok:
    try:
        urllib.request.urlretrieve(ZIP_URL, 'depth_lab.zip')
        with zipfile.ZipFile('depth_lab.zip') as z:
            z.extractall('.')
        ok = Path('train.py').is_file() and Path('prepare_quality_data.py').is_file()
        print('lab zip OK', ok)
    except Exception as e:
        print('zip fetch failed:', e)
if not ok and IN_COLAB:
    from google.colab import files
    print('Upload depth_lab.zip')
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('.')
    ok = Path('train.py').is_file()
assert ok, 'lab zip missing train.py / prepare_quality_data.py'
_link_to_drive('dist')
_link_to_drive('data')


In [ ]:
from pathlib import Path
import subprocess, sys
print('prepare_quality_data (skips if data/READY already present)…')
# Use subprocess — Colab get_ipython().system() often returns None even on success.
rc = subprocess.call([sys.executable, '-u', 'prepare_quality_data.py', '--data', './data'])
assert rc == 0, f'prepare_quality_data failed rc={rc}'
assert (Path('data') / 'READY').is_file(), 'data/READY missing'
print('data READY ok')


In [ ]:
from pathlib import Path
import hashlib
import subprocess, sys
import tensorflow as tf
USE_DRIVE = globals().get('USE_DRIVE', False)
IN_COLAB = globals().get('IN_COLAB', False)
OUT = Path('/content/drive/MyDrive/pixijoy_ml/depth/dist/depth.keras') if USE_DRIVE else Path('dist') / 'depth.keras'
OUT.parent.mkdir(parents=True, exist_ok=True)
ckpt = OUT.with_suffix('.ckpt.keras')
print('ckpt exists' if ckpt.is_file() else 'fresh train', OUT)

gpus = tf.config.list_physical_devices('GPU')
batch_flag = [] if gpus else ['--batch', '4']
if not gpus:
    print('CPU/laptop: batch=4 — slow but valid quality train; resume after interrupt')

cmd = [sys.executable, '-u', 'train.py', '--quality', '--out', str(OUT)] + batch_flag
if ckpt.is_file():
    cmd.append('--resume')
    print('will resume from checkpoint')
print('RUN', ' '.join(cmd))
rc = subprocess.call(cmd)
assert rc == 0, f'train failed rc={rc}'
rc = subprocess.call([sys.executable, '-u', 'export_tflite.py', '--keras', str(OUT), '--out', 'dist/depth.tflite'])
assert rc == 0, f'export failed rc={rc}'
rc = subprocess.call([sys.executable, '-u', 'verify_contract.py', '--quality', '--data', './data'])
assert rc == 0, f'verify --quality failed rc={rc} — do NOT pin'
p = Path('dist/depth.tflite')
assert p.is_file(), p
print('SHIP_OK', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from pathlib import Path
p = Path('dist/depth.tflite')
print('artifact', p.resolve(), 'bytes', p.stat().st_size)
try:
    from google.colab import files
    files.download('dist/depth.tflite')
except Exception:
    print('Not Colab — copy the TFLite from dist/ yourself')
print('Attach depth.tflite to ml-packs-v1 ONLY if verify printed OK / SHIP_OK')
